# Amphibian habitat and protected areas in Québec
**Question the map answers:** which recorded amphibian habitats have the largest share of their area inside a protected area?

This notebook reads three datasets, measures how much of each habitat is protected, and writes the files the web page loads into `data/`:
`amphibians.geojson`, `protected_areas.geojson`, `amphibian_protected_overlap.geojson` and `meta.json`.
Run it top to bottom (Run All). The last cell is the check cell and should print `CHECK PASSED`.

**Scope.** It keeps the regions in `STUDY_REGIONS` (Outaouais and Montérégie). If any file is bigger than `MAX_MB` it simplifies the shapes a little more, and if the files are still too big it falls back to Montérégie only. The settings are in the next cell.

In [1]:
import json
import os

import geopandas as gpd
import pyogrio

# ---------- SETTINGS (change them here) ----------
SCOPE = "auto"             # "auto": all of Québec if the files are small enough, otherwise Montérégie
                           # "quebec": force all of Québec      "monteregie": force Montérégie only
PROTECTED_SCOPE = "near"       # which protected areas go on the map:
                               # "near": only those within PROTECTED_NEAR_KM of a recorded habitat (the frogs live in a small part of Québec)
                               # "matching": only those that overlap a habitat      "all": every protected area in the scope (big files)
PROTECTED_NEAR_KM = 10         # for "near": how far from a habitat a protected area can be and still be shown, in kilometres
STUDY_REGIONS = ["Montérégie", "Outaouais"]   # the map only covers these administrative regions (RES_NM_REG in SDA.gpkg).
                               # Outaouais includes Gatineau, the Pontiac (Shawville, Île du Grand Calumet) and the Collines-de-l'Outaouais.
# Which species go on the map, and which Québec legal statuses to keep for each one.
# The status is the LOIEMV column: Menacée = threatened, Vulnérable = vulnerable,
# Susceptible = likely to be designated threatened or vulnerable.
# Species are matched on the start of the scientific name. Use None to keep every amphibian again.
SPECIES_STATUS = {
    "Pseudacris triseriata": ["Menacée", "Vulnérable"],                    # Western chorus frog
}
MAX_MB = 10                # the biggest file the page should have to load, in megabytes
MIN_OVERLAP_M2 = 100       # overlaps smaller than this (100 square metres) are map slivers, not protection
TOLERANCES = [0.0005, 0.001, 0.002, 0.004]   # how much shapes are simplified, in degrees (0.001 is about 100 m)

# Part 1: Amphibian records
**Step 1: Load the species data**

In [2]:
print(pyogrio.list_layers("../data/emvs_dq.gpkg"))

gdf = gpd.read_file("../data/emvs_dq.gpkg", layer="EMVS_publique")
print(f"All species records: {len(gdf)}")

[['EMVS_publique' 'MultiPolygon']]
All species records: 13298


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: Non-conformant content for record 1 in column EORANKDATE, 2023-11-02T00:00:00.0Z, successfully parsed
  return ogr_read(


**Step 2: Keep only amphibians** (matched on the scientific name, whole words only, so that plants like *Notothylas* are not matched by *Hyla*)

In [3]:
amphibian_genera = [
    "Rana", "Lithobates", "Pseudacris", "Hyla",
    "Anaxyrus", "Bufo",
    "Ambystoma", "Plethodon", "Desmognathus",
    "Notophthalmus", "Necturus", "Eurycea",
    "Gyrinophilus", "Hemidactylium", "Dryophytes"
]
pattern = r"\b(?:" + "|".join(amphibian_genera) + r")\b"
amphibian_mask = gdf["SNAME"].str.contains(pattern, case=False, na=False, regex=True)
amphibians = gdf[amphibian_mask].copy()
print(amphibians["LOIEMV"].value_counts(dropna=False))

# For information: every amphibian species that Québec lists as Menacée or Vulnérable in this data
endangered_all = amphibians[amphibians["LOIEMV"].isin(["Menacée", "Vulnérable"])]
print("\nAll amphibians that are Menacée or Vulnérable (not only the ones on the map):")
print(endangered_all.groupby(["SNAME", "GCOMNAME", "LOIEMV"]).size())
print()

if SPECIES_STATUS:                                    # keep only the species chosen in the settings
    chosen = amphibians[amphibians["SNAME"].str.startswith(tuple(SPECIES_STATUS), na=False)].copy()
    print("\nRecords of the chosen species, by Québec status:")
    print(chosen.groupby(["SNAME", "LOIEMV"]).size())

    def wanted(row):                                  # keep the record only if its status is one chosen for its species
        for name, statuses in SPECIES_STATUS.items():
            if row["SNAME"].startswith(name):
                return row["LOIEMV"] in statuses
        return False

    amphibians = chosen[chosen.apply(wanted, axis=1)].copy()
    print(f"\nKept the chosen species and statuses: {len(amphibians)} records")
print(f"Records before: {len(gdf)}, amphibians kept: {len(amphibians)}, removed: {len(gdf) - len(amphibians)}")
print(amphibians["SNAME"].value_counts())


LOIEMV
Susceptible    934
Vulnérable     399
Menacée        172
Name: count, dtype: int64

All amphibians that are Menacée or Vulnérable (not only the ones on the map):
SNAME                              GCOMNAME                                                       LOIEMV    
Desmognathus ochrophaeus pop. 2    Allegheny Mountain Dusky Salamander - Appalachian population   Menacée        17
Gyrinophilus porphyriticus pop. 2  Spring Salamander - Adirondack / Appalachian population        Vulnérable    399
Pseudacris triseriata              Western Chorus Frog                                            Menacée       155
dtype: int64


Records of the chosen species, by Québec status:
SNAME                  LOIEMV 
Pseudacris triseriata  Menacée    155
dtype: int64

Kept the chosen species and statuses: 155 records
Records before: 13298, amphibians kept: 155, removed: 13143
SNAME
Pseudacris triseriata    155
Name: count, dtype: int64


**Step 3: Keep the metres.** The areas are measured in metres (EPSG:32198, NAD83 / Québec Lambert), because degrees cannot give hectares.
The shapes are converted to latitude and longitude (EPSG:4326, what Leaflet uses) only when they are written, in Part 6.

In [4]:
print("Working in:", amphibians.crs)
amphibians["geometry"] = amphibians.geometry.make_valid()
amphibians["rec_id"] = range(len(amphibians))      # a simple id for each record, used by the search box

Working in: EPSG:32198


# Part 2: The study area
The regions chosen in `STUDY_REGIONS` (Outaouais and Montérégie) cover Gatineau, Shawville, Île du Grand Calumet and Montérégie. Habitat records and protected areas outside them are dropped. The Montérégie boundary is also used if the notebook has to fall back to Montérégie only.

In [5]:
print(pyogrio.list_layers("../data/SDA.gpkg"))
regions = gpd.read_file("../data/SDA.gpkg", layer="regio_s")

monteregie = regions[regions["RES_NM_REG"].str.contains("Montérégie", case=False, na=False)]
print(f"Montérégie polygon(s) found: {len(monteregie)}")

monteregie_boundary = monteregie.to_crs(amphibians.crs).union_all()      # the boundary is in EPSG:4269
print("Boundary bounds (metres):", monteregie_boundary.bounds)

# The study area: the regions chosen in the settings. Habitat records and protected areas outside it are dropped.
study = regions[regions["RES_NM_REG"].str.contains("|".join(STUDY_REGIONS), case=False, na=False)]
print("Regions found:", sorted(study["RES_NM_REG"]))
for name in STUDY_REGIONS:
    assert study["RES_NM_REG"].str.contains(name, case=False, na=False).any(), f"region not found: {name} (check STUDY_REGIONS)"
study_area = study.to_crs(amphibians.crs).union_all()

n_before = len(amphibians)
amphibians = amphibians[amphibians.intersects(study_area)].copy()
amphibians["rec_id"] = range(len(amphibians))          # new ids, since some records were dropped
print(f"Habitat records inside the study area: {len(amphibians)} of {n_before} (dropped {n_before - len(amphibians)})")


[['arron_s' 'MultiPolygon Z']
 ['comet_s' 'MultiPolygon Z']
 ['arron_l' 'LineString Z']
 ['comet_l' 'LineString Z']
 ['mrc_l' 'LineString Z']
 ['munic_l' 'LineString Z']
 ['regio_l' 'LineString Z']
 ['mrc_s' 'MultiPolygon Z']
 ['munic_s' 'MultiPolygon Z']
 ['regio_s' 'MultiPolygon Z']]
Montérégie polygon(s) found: 1
Boundary bounds (metres): (-486868.29111281224, 124980.57390394984, -297800.1203513101, 246335.4500177664)
Regions found: ['Montérégie', 'Outaouais']
Habitat records inside the study area: 155 of 155 (dropped 0)


# Part 3: Protected areas
**Step 1: Load the registry** (Registre des aires protégées et des AMCE, layer `AP_REG_S`)

In [6]:
reg = "../data/registre_aires_prot_GPKG/registre_aires_prot.gpkg"
print(pyogrio.list_layers(reg))

protected = gpd.read_file(reg, layer="AP_REG_S").to_crs(amphibians.crs)
print(f"Protected areas in Québec: {len(protected)}")

[['AP_REG_S' 'MultiPolygon']
 ['AP_ZON_S' 'MultiPolygon']
 ['T_IMP_S' 'MultiPolygon']]
Protected areas in Québec: 5611


**Step 2: Keep only designations that protect general habitat.**
I remove areas designated for plants, birds, muskrats or herons, because they are not aimed at amphibians.

In [7]:
n0 = len(protected)

# plant habitat designations
protected_clean = protected[~protected["DESIGNOM"].str.contains("floristique", case=False, na=False)]
n1 = len(protected_clean)

# bird, muskrat and heron designations
protected_clean = protected_clean[
    ~protected_clean["DESIGNOM"].str.contains("oiseaux|rat musqu|Héronni", case=False, na=False)
].copy()
n2 = len(protected_clean)

protected_clean["geometry"] = protected_clean.geometry.make_valid()
print(f"Before: {n0}, after removing plant habitat: {n1}, after removing bird, muskrat and heron habitat: {n2}")
print(protected_clean["DESIGNOM"].value_counts())

Before: 5611, after removing plant habitat: 5582, after removing bird, muskrat and heron habitat: 4547
DESIGNOM
Refuge biologique                                          3110
Milieu naturel de conservation volontaire                   471
Réserve naturelle reconnue                                  271
Forêt ancienne                                              152
Réserve de territoire aux fins d'aire protégée               84
Réserve écologique                                           72
Forêt rare                                                   72
Réserve de biodiversité projetée                             71
Aire de confinement du cerf de Virginie                      52
Habitat d'une espèce faunique menacée ou vulnérable          30
Territoire mis en réserve                                    29
Parc national                                                28
Autres mesures de conservation efficaces                     20
Forêt refuge                                            

**Step 3: Add the Emergency Order lots (Longueuil)**
The Canada Gazette Emergency Order SOR/2021-231 protects Western chorus frog habitat in Longueuil. The Order lists Québec cadastre lot numbers, not shapes, so the 21 whole lots were downloaded from the Québec cadastre layer (`Lots du cadastre rénové`, field `NO_LOT`) and saved as `data/longueuil_order_lots.geojson`. They are merged into one protected area and added to the registry areas before the protection is measured. The 27 partial parcels in the Order are not included.

In [8]:
import os
print("The notebook is running in:", os.getcwd())
print("It is looking for:", os.path.abspath("../data/longueuil_order_lots.geojson"))
print("Does that file exist?", os.path.exists("../data/longueuil_order_lots.geojson"))
print("Files in data/:")
for f in sorted(os.listdir("../data")):
    print("  ", f)

The notebook is running in: /Users/myast-james/Desktop/GEOG464/Term-project/notebooks
It is looking for: /Users/myast-james/Desktop/GEOG464/Term-project/data/longueuil_order_lots.geojson
Does that file exist? True
Files in data/:
   SDA.gpkg
   amphibian_protected_overlap.geojson
   amphibians.geojson
   emvs_dq.gpkg
   frog_protected_overlap.geojson
   longueuil_order_lots.geojson
   meta.json
   protected_areas.geojson
   protected_monteregie.geojson
   registre_aires_prot_GPKG


In [9]:
import pandas as pd

ORDER_FILE = "../data/longueuil_order_lots.geojson"
if os.path.exists(ORDER_FILE):
    lots = gpd.read_file(ORDER_FILE).to_crs(amphibians.crs)
    print(f"Emergency Order lots read: {len(lots)} (the Order names 21 whole lots)")
    lots["geometry"] = lots.geometry.make_valid()
    order_area = gpd.GeoDataFrame(
        {"NOM": ["Emergency Order, Longueuil (SOR/2021-231)"],
         "DESIGNOM": ["Ordonnance d'urgence (LEP)"],
         "DA_CREATIO": ["2021-11-20"]},
        geometry=[lots.union_all()], crs=amphibians.crs)
    if str(protected_clean["DA_CREATIO"].dtype).startswith("datetime"):       # same type as the registry column
        order_area["DA_CREATIO"] = pd.to_datetime(order_area["DA_CREATIO"])
    protected_clean = pd.concat([protected_clean, order_area], ignore_index=True)
    print(f"Protected areas now: {len(protected_clean)} (one of them is the Emergency Order)")
else:
    print("No Emergency Order file found in data/: skipped.")

Emergency Order lots read: 21 (the Order names 21 whole lots)
Protected areas now: 4548 (one of them is the Emergency Order)


# Part 4: How much of each habitat is protected?
For every amphibian record this measures the habitat's area, the part of it inside a protected area, and the percentage.
Where protected areas overlap each other, the area is counted once (they are merged first).

**Step 1: Measure each habitat**

In [10]:
sindex = protected_clean.sindex

habitat_ha, protected_ha, names = [], [], []
for geom in amphibians.geometry:
    candidates = protected_clean.iloc[sindex.query(geom, predicate="intersects")]
    # keep only the areas that really share area with the habitat (touching along an edge, or a sliver under MIN_OVERLAP_M2, does not count)
    real = candidates[[geom.intersection(g).area > MIN_OVERLAP_M2 for g in candidates.geometry]]
    if len(real):
        inside = geom.intersection(real.geometry.union_all())
        protected_ha.append(inside.area / 10000)
        names.append("; ".join(sorted(set(real["NOM"].astype(str)))))
    else:
        protected_ha.append(0.0)
        names.append(None)
    habitat_ha.append(geom.area / 10000)

amphibians["habitat_ha"] = [round(v, 2) for v in habitat_ha]
amphibians["protected_ha"] = [round(v, 2) for v in protected_ha]
amphibians["pct_protected"] = [round(min(100.0, 100 * p / h), 1) if h > 0 else 0.0 for p, h in zip(protected_ha, habitat_ha)]
amphibians["protected_by"] = names
amphibians["is_protected"] = amphibians["protected_by"].notna()

print(amphibians.groupby("SNAME")["is_protected"].agg(protected="sum", records="count"))
print(amphibians[["habitat_ha", "protected_ha", "pct_protected"]].describe())

# How far is the nearest protected area? (most useful for the habitat that is not protected)
nearest = gpd.sjoin_nearest(amphibians[["geometry"]], protected_clean[["NOM", "geometry"]], how="left", distance_col="nearest_m")
nearest = nearest[~nearest.index.duplicated(keep="first")]          # if two areas are equally near, keep one
amphibians["nearest_name"] = nearest["NOM"].reindex(amphibians.index)
amphibians["nearest_km"] = (nearest["nearest_m"].reindex(amphibians.index) / 1000).round(1)
print(amphibians.loc[~amphibians["is_protected"], "nearest_km"].describe())


                       protected  records
SNAME                                    
Pseudacris triseriata         25      155
       habitat_ha  protected_ha  pct_protected
count  155.000000    155.000000     155.000000
mean    76.784065      5.625290       7.399355
std    135.730540     25.736701      22.848691
min      0.010000      0.000000       0.000000
25%     12.580000      0.000000       0.000000
50%     24.510000      0.000000       0.000000
75%     68.920000      0.000000       0.000000
max    821.560000    267.530000     100.000000
count    130.000000
mean       2.908462
std        2.059993
min        0.100000
25%        1.300000
50%        2.600000
75%        3.875000
max       10.900000
Name: nearest_km, dtype: float64


**Step 2: The ranking the page shows.** The search box on the page does this same sort. Here are the top ten, so you can compare.

In [11]:
top = (amphibians[amphibians["is_protected"]]
       .sort_values(["pct_protected", "protected_ha"], ascending=False)
       .head(10))
print(top[["SNAME", "habitat_ha", "protected_ha", "pct_protected", "protected_by"]].to_string(index=False))

                SNAME  habitat_ha  protected_ha  pct_protected                                                                                                      protected_by
Pseudacris triseriata       31.62         31.62          100.0                                                                                                          Gatineau
Pseudacris triseriata       25.17         25.17          100.0                                                                                                          Gatineau
Pseudacris triseriata       12.58         12.58          100.0                                                                                                          Gatineau
Pseudacris triseriata        0.01          0.01          100.0                                                                                                          Gatineau
Pseudacris triseriata        9.21          9.07           98.5                                                     

**Step 3: Cut out the overlap** (the part of each amphibian habitat that lies inside a protected area, for every species)

In [12]:
overlap = gpd.overlay(
    amphibians[["rec_id", "SNAME", "GCOMNAME", "SCOMNAME", "LOIEMV", "geometry"]],
    protected_clean[["NOM", "DESIGNOM", "DA_CREATIO", "geometry"]],
    how="intersection",
    keep_geom_type=True,
)
overlap = overlap[overlap.geometry.area > MIN_OVERLAP_M2]         # drop slivers, as in Step 1
print(f"{int(amphibians['is_protected'].sum())} of {len(amphibians)} amphibian records overlap a protected area; {len(overlap)} overlap pieces")

25 of 155 amphibian records overlap a protected area; 39 overlap pieces


**Step 4: Which protected areas go on the map?**
The frogs live in a small part of Québec, so most protected areas are far from any habitat and only slow the page down.
With `PROTECTED_SCOPE = "near"` only the protected areas within `PROTECTED_NEAR_KM` kilometres of a recorded habitat are kept, which shows the habitat and its surroundings.
With `"matching"` only the areas that overlap a habitat are kept, and with `"all"` every protected area is kept.

In [13]:
in_study = protected_clean[protected_clean.intersects(study_area)]          # first: only the protected areas inside the study area
if PROTECTED_SCOPE == "near":
    # then: only those within PROTECTED_NEAR_KM of any habitat (measured in metres, in EPSG:32198)
    near_zone = amphibians.geometry.buffer(PROTECTED_NEAR_KM * 1000).union_all()
    protected_map = in_study[in_study.intersects(near_zone)]
elif PROTECTED_SCOPE == "matching":
    keep = in_study.index.isin(overlap.merge(protected_clean[["NOM", "DESIGNOM", "DA_CREATIO"]].reset_index(),
                                             on=["NOM", "DESIGNOM", "DA_CREATIO"])["index"])
    protected_map = in_study[keep]
else:
    protected_map = in_study                                                   # every protected area in the study area
print(f"Protected areas on the map: {len(protected_map)} of {len(protected_clean)} (inside the study area: {len(in_study)})")


Protected areas on the map: 119 of 4548 (inside the study area: 683)


# Part 5: Choose the scope and write the files for the web page
Shapes are simplified a little so the files stay small, and converted to latitude and longitude. The notebook starts with all of Québec and only falls back to Montérégie if the files are still bigger than `MAX_MB`.

In [14]:
amph_cols = ["rec_id", "SNAME", "SCOMNAME", "GCOMNAME", "LOIEMV", "LASTOBS",
             "habitat_ha", "protected_ha", "pct_protected", "protected_by", "is_protected",
             "nearest_name", "nearest_km", "geometry"]
amph_out = amphibians[[c for c in amph_cols if c in amphibians.columns]]
prot_out = protected_map[["NOM", "DESIGNOM", "DA_CREATIO", "geometry"]]
ovl_out = overlap.drop(columns=["rec_id"])

def simplify(gdf, tol):
    out = gdf.to_crs(epsg=4326)
    out["geometry"] = out.geometry.simplify(tol, preserve_topology=True)
    return out[~out.geometry.is_empty]

def write_geojson(gdf, path):
    if len(gdf) == 0:      # an empty file keeps the page from failing to load
        with open(path, "w") as f:
            f.write('{"type":"FeatureCollection","features":[]}')
    else:
        gdf.to_file(path, driver="GeoJSON", COORDINATE_PRECISION=5)
    return round(os.path.getsize(path) / 1e6, 2)

def export(scope, tol):
    a, p, o = amph_out, prot_out, ovl_out
    if scope == "monteregie":
        a = a[a.within(monteregie_boundary)]
        p = p[p.intersects(monteregie_boundary)]
        o = overlap[overlap["rec_id"].isin(a["rec_id"])].drop(columns=["rec_id"])
    sizes = {
        "amphibians": write_geojson(simplify(a, tol), "../data/amphibians.geojson"),
        "protected_areas": write_geojson(simplify(p, tol), "../data/protected_areas.geojson"),
        "overlap": write_geojson(simplify(o, tol), "../data/amphibian_protected_overlap.geojson"),
    }
    return a, p, o, sizes

chosen = None
if SCOPE in ("auto", "quebec"):
    for tol in TOLERANCES:
        a, p, o, sizes = export("quebec", tol)
        print(f"Québec, simplify {tol}: {sizes} MB")
        if max(sizes.values()) <= MAX_MB:
            chosen = ("quebec", tol)
            break
    if chosen is None and SCOPE == "quebec":
        chosen = ("quebec", TOLERANCES[-1])          # forced: keep it even if it is big
if chosen is None:
    a, p, o, sizes = export("monteregie", TOLERANCES[0])
    chosen = ("monteregie", TOLERANCES[0])
    print(f"Montérégie, simplify {TOLERANCES[0]}: {sizes} MB")

scope, tol = chosen
meta = {
    "scope": scope,
    "protected_scope": PROTECTED_SCOPE,
    "protected_near_km": PROTECTED_NEAR_KM,
    "regions": STUDY_REGIONS,
    "simplify_degrees": tol,
    "amphibian_records": len(a),
    "species": int(a["SNAME"].nunique()),
    "protected_records": int(a["is_protected"].sum()),
    "protected_areas": len(p),
    "overlap_pieces": len(o),
    "sizes_mb": sizes,
}
with open("../data/meta.json", "w") as f:
    json.dump(meta, f, indent=1)

print("\nFINAL SCOPE:", scope)
print(json.dumps(meta, indent=1))

Québec, simplify 0.0005: {'amphibians': 0.16, 'protected_areas': 0.1, 'overlap': 0.03} MB

FINAL SCOPE: quebec
{
 "scope": "quebec",
 "protected_scope": "near",
 "protected_near_km": 10,
 "regions": [
  "Mont\u00e9r\u00e9gie",
  "Outaouais"
 ],
 "simplify_degrees": 0.0005,
 "amphibian_records": 155,
 "species": 1,
 "protected_records": 25,
 "protected_areas": 119,
 "overlap_pieces": 39,
 "sizes_mb": {
  "amphibians": 0.16,
  "protected_areas": 0.1,
  "overlap": 0.03
 }
}


# Check cell
Tests the files the page loads: count, projection, area, size, properties and the protected-share numbers. It must print `CHECK PASSED`.

In [15]:
meta = json.load(open("../data/meta.json"))
check = gpd.read_file("../data/amphibians.geojson")
areas = gpd.read_file("../data/protected_areas.geojson")
print(meta["scope"], "|", len(check), "records;", check["SNAME"].nunique(), "species;", len(areas), "protected areas;",
      check.crs, check.total_bounds)

# count: what the notebook said it wrote is what is in the file
assert len(check) == meta["amphibian_records"], "record count differs from meta.json"
assert len(areas) == meta["protected_areas"], "protected area count differs from meta.json"
assert check["rec_id"].is_unique, "rec_id must be unique (the search box uses it)"

# projection
assert check.crs.to_epsg() == 4326 and areas.crs.to_epsg() == 4326, "files should be in latitude and longitude (EPSG:4326)"

# area: every record must sit inside the scope
minx, miny, maxx, maxy = check.total_bounds
if meta["scope"] == "monteregie":
    assert -74.7 < minx and maxx < -72.3 and 44.9 < miny and maxy < 46.2, "records outside the Montérégie area"
else:
    assert -80 < minx and maxx < -57 and 44.9 < miny and maxy < 63, "records outside Québec"

# size: each file stays small enough for the page
for name, mb in meta["sizes_mb"].items():
    assert mb <= MAX_MB or SCOPE == "quebec", name + " is bigger than MAX_MB"

# properties the page uses
for col in ["rec_id", "GCOMNAME", "SNAME", "LOIEMV", "habitat_ha", "protected_ha", "pct_protected", "protected_by", "is_protected", "nearest_km"]:
    assert col in check.columns, "missing column: " + col
for col in ["NOM", "DESIGNOM", "DA_CREATIO"]:
    assert col in areas.columns, "missing column: " + col

# the protected-share numbers make sense
assert check["pct_protected"].between(0, 100).all(), "a percentage is outside 0 to 100"
assert (check["protected_ha"] <= check["habitat_ha"] * 1.01 + 0.01).all(), "more protected area than habitat area"
assert (check.loc[~check["is_protected"].astype(bool), "pct_protected"] == 0).all(), "an unprotected record has a percentage"
assert (check.loc[check["is_protected"].astype(bool), "protected_ha"] > 0).all(), "a protected record has no protected hectares"

# the headline and the overlap layer must agree
n_protected = int(check["is_protected"].astype(bool).sum())
assert n_protected == meta["protected_records"], "protected count differs from meta.json"
assert (n_protected > 0) == (meta["overlap_pieces"] > 0), "is_protected and the overlap file disagree"

print(f"{n_protected} of {len(check)} amphibian records overlap a protected area")
print("CHECK PASSED")

quebec | 155 records; 1 species; 119 protected areas; EPSG:4326 [-77.07646  45.09659 -73.1953   45.85652]
25 of 155 amphibian records overlap a protected area
CHECK PASSED
